# LDA — Template para Novo Corpus

Copie este notebook para `notebooks/lda/02_lda_<corpus>.ipynb` e substitua os
placeholders marcados com `# <<` para adaptar a um novo corpus.

## Pipeline (5 etapas)
1. **Lematização** — spaCy (`pt_core_news_lg` ou `en_core_web_sm`) via `_helpers.lemmatize_corpus`
2. **Vocabulário** — `gensim.Dictionary`, `no_below=5`, `no_above=0.5` → BoW
3. **Seleção de K** — grid search `K ∈ [k_min, k_max]`, maximiza coerência **C_v**
4. **Modelo final** — `LdaMulticore` com `K*` e 20 passes
5. **Atribuição** — `argmax(θ)` por documento → `lda_results.csv`

## Saídas
- `lda_results.csv` — `post_id, topic_id, topic_name, topic_prob_distribution`
- `lda_topics_for_eval.csv` — `topic_id, topic_name, keywords` (comparativo cross-model)
- `lda_coherence_curve.png`, `lda_wordclouds.png`, `lda_pyldavis.html`


In [ ]:
# Descomente se necessário
# !pip install gensim pyLDAvis spacy wordcloud -q
# !python -m spacy download pt_core_news_lg -q
# !python -m spacy download en_core_web_sm -q


In [ ]:
# ── path fix: resolve _helpers.py e data/ de qualquer subdiretório ──
import sys as _sys, os as _os
from pathlib import Path as _P
_here = _P().resolve()
_nb_dir = _here if (_here / '_helpers.py').exists() else _here.parent
if str(_nb_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_dir))
_os.chdir(_nb_dir)          # '../data/...' paths ficam corretos
del _here, _nb_dir, _P, _sys, _os
# ─────────────────────────────────────────────────────────────────────
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyLDAvis
import pyLDAvis.gensim_models

from _helpers import load_params, get_corpus_config, get_column_names, get_seed, load_corpus
from _helpers import lemmatize_corpus
from _helpers import grid_search_k, grid_search_alpha_eta, train_lda, extract_topics_keywords, compute_doc_distributions
from _helpers import name_all_topics
from _helpers import compute_coherence_cv, export_results, export_topics_for_eval

# ── CONFIGURAÇÃO — edite aqui para cada corpus ──────────────────────────
CORPUS_ID = "social"           # << nome do corpus no params.yaml
# ────────────────────────────────────────────────────────────────────────

params = load_params()
corpus_id, corpus_cfg = get_corpus_config(params, CORPUS_ID)
cols = get_column_names(corpus_cfg)
SEED = get_seed(params)
np.random.seed(SEED)

LANG     = corpus_cfg.get("language", "pt")     # "pt" ou "en" — usado em lematização/naming
TEXT_COL = cols["text"]
ID_COL   = cols["post_id"]

subdir     = corpus_cfg.get("subdir", corpus_id)
INPUT_DIR  = Path(f"../data/input/{subdir}")
from _helpers import make_run_output_dir
BASE_OUTPUT_DIR = Path(f"../data/output/{subdir}")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Saida carimbada por execucao: <corpus>_<data>_<hora> (nao sobrescreve runs anteriores)
OUTPUT_DIR = make_run_output_dir(BASE_OUTPUT_DIR, corpus_id)

lda_params   = params["lda"]
k_min, k_max = lda_params["k_range"]

print(f"Corpus  : {corpus_id}  |  Idioma: {LANG}")
print(f"K range : [{k_min}, {k_max}]")

In [ ]:
df       = load_corpus(str(INPUT_DIR))
docs     = df[TEXT_COL].astype(str).tolist()
post_ids = df[ID_COL].astype(str).tolist() if ID_COL in df.columns \
           else [f"doc_{i:04d}" for i in range(len(df))]

print(f"Docs carregados: {len(docs)}")
print(f"Exemplo: {docs[0][:120]}")

## Lematização (`_helpers.lemmatize_corpus`)

Tokeniza com spaCy (`pt_core_news_lg` ou `en_core_web_sm`, escolhido automaticamente
por `LANG`), remove stop words, tokens não-alfa e tokens com `len(lemma) <= 2`.
Constrói o `gensim.Dictionary` já com `filter_extremes(no_below, no_above)` aplicado
(parâmetros em `params.yaml > lda`).

In [ ]:
print(f"Lematizando (spaCy, lang={LANG})...")
t0 = time.time()
tokenized, dictionary = lemmatize_corpus(docs, LANG, params)
print(f"  concluido em {time.time()-t0:.1f}s")
print(f"  vocabulario apos filter_extremes: {len(dictionary):,} tokens")
print(f"  Exemplo: {tokenized[0][:20]}")

In [ ]:
corpus_bow = [dictionary.doc2bow(d) for d in tokenized]
non_empty  = sum(1 for d in corpus_bow if len(d) > 0)
print(f"Docs com BoW nao-vazio: {non_empty}/{len(corpus_bow)}")

## Selecao de K (grid search C_v)

Para cada `K ∈ [k_min, k_max]` treina `LdaMulticore` (10 passes) e calcula a coerencia **C_v**.
**Atencao:** C_v sistematicamente penaliza K alto (Stevens et al. 2012).
Use o K* automatico como ponto de partida, mas valide com inspecao qualitativa.


In [ ]:
print(f"Grid search K de {k_min} a {k_max} (passes=10)...")
t0 = time.time()
scores, perplexity_scores = grid_search_k(
    corpus_bow, dictionary, tokenized, range(k_min, k_max + 1), seed=SEED, passes=10)
print(f"  concluido em {time.time()-t0:.0f}s")

for k in sorted(scores):
    print(f"  K={k:3d}  c_v={scores[k]:.4f}  perp={perplexity_scores[k]:.1f}")

best_k = max(scores, key=scores.get)
print(f"\nK* automatico (max c_v): {best_k}  (c_v={scores[best_k]:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ks = sorted(scores)
ax.plot(ks, [scores[k] for k in ks], marker="o", linewidth=2, color="steelblue")
ax.axvline(best_k, color="red", linestyle="--", label=f"K*={best_k}")
ax.set_xlabel("K")
ax.set_ylabel("Coerencia C_v")
ax.set_title(f"Grid search K — {corpus_id}")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_coherence_curve.png", dpi=150)
plt.show()

## Modelo final

Ajuste `BEST_K` se a inspecao qualitativa sugerir um K diferente do automatico.


In [ ]:
BEST_K = best_k  # << altere se necessario apos inspecao qualitativa
top_n  = params["evaluation"]["top_n_keywords"]

print(f"Treinando LDA final com K={BEST_K}, 20 passes...")
t0 = time.time()
model = train_lda(corpus_bow, dictionary, k=BEST_K, seed=SEED, passes=20)
print(f"  concluido em {time.time()-t0:.1f}s")

topics_keywords = extract_topics_keywords(model, k=BEST_K, top_n=top_n)
for tid, kws in topics_keywords.items():
    print(f"  T{tid:02d}: {', '.join(kws[:8])}")

In [ ]:
try:
    from wordcloud import WordCloud
    ncols = 3
    nrows = (BEST_K + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows))
    axes = axes.flatten()
    for i in range(BEST_K):
        ws = dict(model.show_topic(i, topn=30))
        wc = WordCloud(width=400, height=200, background_color="white",
                       max_words=25, colormap="Blues").generate_from_frequencies(ws)
        axes[i].imshow(wc, interpolation="bilinear")
        axes[i].axis("off")
        axes[i].set_title(f"T{i}", fontsize=10)
    for j in range(BEST_K, len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"Wordclouds por topico — {corpus_id}  (K={BEST_K})", y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "lda_wordclouds.png", dpi=150, bbox_inches="tight")
    plt.show()
except ImportError:
    print("wordcloud nao instalado — pip install wordcloud")

In [ ]:
llm_cfg = params.get("llm", {})
print(f"Nomeando {BEST_K} tópicos via {llm_cfg.get('model')} (lang={LANG})...")
t0 = time.time()
try:
    topics_names = name_all_topics(
        topics_keywords,
        model=llm_cfg["model"],
        base_url=llm_cfg.get("base_url", "http://localhost:11434"),
        lang=LANG,
    )
    print(f"done em {time.time()-t0:.0f}s")
except Exception as e:
    print(f"LLM falhou ({e}) — fallback top-3 keywords")
    topics_names = {tid: ", ".join(kws[:3]) for tid, kws in topics_keywords.items()}

for tid, name in topics_names.items():
    print(f"  T{tid:02d}: {name}")

In [ ]:
dominant, full = compute_doc_distributions(model, corpus_bow, k=BEST_K)

counts = pd.Series(dominant).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(counts.index, counts.values, color="steelblue")
ax.set_xlabel("Topico")
ax.set_ylabel("Documentos")
ax.set_title(f"Distribuicao docs por topico — {corpus_id}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_distribution.png", dpi=150)
plt.show()

cv_final = compute_coherence_cv(topics_keywords, tokenized, dictionary)
print(f"C_v final (K={BEST_K}): {cv_final:.4f}")

In [ ]:
try:
    vis = pyLDAvis.gensim_models.prepare(model, corpus_bow, dictionary)
    pyLDAvis.save_html(vis, str(OUTPUT_DIR / "lda_pyldavis.html"))
    print(f"pyLDAvis salvo em {OUTPUT_DIR / 'lda_pyldavis.html'}")
    pyLDAvis.display(vis)
except Exception as e:
    print(f"pyLDAvis indisponivel: {e}")


In [ ]:
export_results(
    topics=dominant, probs=full, names=topics_names, texts=docs,
    output_path=str(OUTPUT_DIR / "lda_results.csv"),
    topic_type="probabilistic", granularity="unit", post_ids=post_ids,
)
export_topics_for_eval(
    topics_keywords=topics_keywords, topics_names=topics_names,
    model_name="lda", output_path=str(OUTPUT_DIR / "lda_topics_for_eval.csv"),
)

print(f"Exportado: {OUTPUT_DIR / 'lda_results.csv'}  ({len(docs)} docs)")
print(f"Exportado: {OUTPUT_DIR / 'lda_topics_for_eval.csv'}")